# 第 30 章：神经网络基础

这个 notebook 用一个极小的非线性回归教学数据集演示神经网络最小 workflow：

- 先建立线性回归 baseline
- 再用纯 Python 实现一个单隐层 `tanh` 网络
- 比较两者在测试集上的误差
- 回到隐藏单元激活，理解网络到底补上了什么表达能力

教学说明：这里刻意不依赖 NumPy、PyTorch 或深度学习框架就能跑通，目的是先把网络结构、损失函数和梯度更新的逻辑讲清楚。环境里如果装了 PyTorch，最后一格会给出一个可选对照实现。


In [ ]:
from __future__ import annotations

import csv
import math
from pathlib import Path

DATA_PATH = Path("../../data/small/neural_network_regression_demo.csv").resolve()

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            "sample_id": raw["sample_id"],
            "feature_x": float(raw["feature_x"]),
            "target_y": float(raw["target_y"]),
            "split": raw["split"],
        })

train_rows = [row for row in rows if row["split"] == "train"]
test_rows = [row for row in rows if row["split"] == "test"]

print(f"Loaded {len(rows)} nonlinear-regression samples from {DATA_PATH.name}")
print("train/test sizes:", len(train_rows), len(test_rows))
print("train ids:", [row["sample_id"] for row in train_rows[:4]], "...")
print("test ids:", [row["sample_id"] for row in test_rows])
print("feature_x range:", (min(row["feature_x"] for row in rows), max(row["feature_x"] for row in rows)))
print("target_y range:", (min(row["target_y"] for row in rows), max(row["target_y"] for row in rows)))


In [ ]:
def fit_line(xs, ys):
    x_mean = sum(xs) / len(xs)
    y_mean = sum(ys) / len(ys)
    numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
    denominator = sum((x - x_mean) ** 2 for x in xs)
    slope = numerator / denominator
    intercept = y_mean - slope * x_mean
    return slope, intercept


def mean_absolute_error(y_true, y_pred):
    return sum(abs(a - b) for a, b in zip(y_true, y_pred)) / len(y_true)


def root_mean_squared_error(y_true, y_pred):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(y_true, y_pred)) / len(y_true))

train_xs = [row["feature_x"] for row in train_rows]
train_ys = [row["target_y"] for row in train_rows]
test_xs = [row["feature_x"] for row in test_rows]
test_ys = [row["target_y"] for row in test_rows]

linear_slope, linear_intercept = fit_line(train_xs, train_ys)
linear_predictions = [linear_intercept + linear_slope * x for x in test_xs]

print("linear baseline coefficients:", round(linear_slope, 4), round(linear_intercept, 4))
print("linear test MAE:", round(mean_absolute_error(test_ys, linear_predictions), 4))
print("linear test RMSE:", round(root_mean_squared_error(test_ys, linear_predictions), 4))
print("linear test predictions:")
for row, prediction in zip(test_rows, linear_predictions):
    print(row["sample_id"], "true=", row["target_y"], "pred=", round(prediction, 3), "residual=", round(prediction - row["target_y"], 3))


In [ ]:
feature_mean = sum(train_xs) / len(train_xs)
feature_std = math.sqrt(sum((x - feature_mean) ** 2 for x in train_xs) / len(train_xs)) or 1.0

standardized_train = [((row["feature_x"] - feature_mean) / feature_std, row["target_y"]) for row in train_rows]
standardized_test = [((row["feature_x"] - feature_mean) / feature_std, row["target_y"]) for row in test_rows]

hidden_size = 6
w1 = [-1.4, -0.9, -0.4, 0.2, 0.7, 1.1]
b1 = [-0.6, -0.2, 0.1, -0.1, 0.3, 0.6]
w2 = [-0.5, -0.4, -0.2, 0.2, 0.4, 0.6]
b2 = 0.0
learning_rate = 0.05
training_snapshots = {}


def forward_pass(x_value):
    hidden_pre = [w1[index] * x_value + b1[index] for index in range(hidden_size)]
    hidden_act = [math.tanh(value) for value in hidden_pre]
    prediction = sum(w2[index] * hidden_act[index] for index in range(hidden_size)) + b2
    return hidden_pre, hidden_act, prediction


for epoch in range(6000):
    grad_w1 = [0.0] * hidden_size
    grad_b1 = [0.0] * hidden_size
    grad_w2 = [0.0] * hidden_size
    grad_b2 = 0.0
    mean_loss = 0.0

    for x_value, y_true in standardized_train:
        hidden_pre, hidden_act, prediction = forward_pass(x_value)
        error = prediction - y_true
        mean_loss += 0.5 * error * error

        for index in range(hidden_size):
            grad_w2[index] += error * hidden_act[index]
        grad_b2 += error

        for index in range(hidden_size):
            delta_hidden = error * w2[index] * (1.0 - hidden_act[index] ** 2)
            grad_w1[index] += delta_hidden * x_value
            grad_b1[index] += delta_hidden

    sample_count = len(standardized_train)
    mean_loss /= sample_count

    for index in range(hidden_size):
        w1[index] -= learning_rate * grad_w1[index] / sample_count
        b1[index] -= learning_rate * grad_b1[index] / sample_count
        w2[index] -= learning_rate * grad_w2[index] / sample_count
    b2 -= learning_rate * grad_b2 / sample_count

    if epoch + 1 in {1, 200, 1000, 6000}:
        training_snapshots[epoch + 1] = mean_loss

print("standardization:", {"mean": round(feature_mean, 3), "std": round(feature_std, 3)})
print("training loss snapshots:", {epoch: round(loss_value, 5) for epoch, loss_value in training_snapshots.items()})


In [ ]:
def network_predict(raw_x_value):
    standardized_x = (raw_x_value - feature_mean) / feature_std
    _, hidden_act, prediction = forward_pass(standardized_x)
    return prediction, hidden_act

network_predictions = []
for row in test_rows:
    prediction, _ = network_predict(row["feature_x"])
    network_predictions.append(prediction)

linear_rmse = root_mean_squared_error(test_ys, linear_predictions)
network_rmse = root_mean_squared_error(test_ys, network_predictions)
linear_mae = mean_absolute_error(test_ys, linear_predictions)
network_mae = mean_absolute_error(test_ys, network_predictions)

print("network test MAE:", round(network_mae, 4))
print("network test RMSE:", round(network_rmse, 4))
print("RMSE improvement over linear baseline:", round(linear_rmse - network_rmse, 4))
print("test-set comparison:")
for row, linear_prediction, network_prediction in zip(test_rows, linear_predictions, network_predictions):
    print(
        row["sample_id"],
        "true=", row["target_y"],
        "linear=", round(linear_prediction, 3),
        "network=", round(network_prediction, 3),
        "linear_residual=", round(linear_prediction - row["target_y"], 3),
        "network_residual=", round(network_prediction - row["target_y"], 3),
    )


In [ ]:
probe_points = [-1.5, -0.5, 0.0, 0.8, 1.6]
print("hidden-unit responses at representative x values:")
for point in probe_points:
    prediction, hidden_act = network_predict(point)
    rounded_hidden = [round(value, 3) for value in hidden_act]
    print(f"x={point:+.2f} -> prediction={prediction:.3f}, hidden={rounded_hidden}")

print("Interpretation:")
print("  The linear baseline captures the overall upward trend but misses the saturation at both ends.")
print("  Hidden units respond differently across x, so their weighted combination can bend the final prediction curve.")
print("  This is the first real reason neural networks outperform a straight line here: not magic, but learned nonlinear basis functions.")


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Optional matplotlib plot skipped: {exc}")
else:
    figure, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(train_xs, train_ys, color="tab:blue", label="train")
    ax.scatter(test_xs, test_ys, color="tab:orange", marker="s", label="test")

    grid_xs = [min(train_xs) + 0.05 * index for index in range(int((max(test_xs) - min(train_xs)) / 0.05) + 1)]
    linear_grid = [linear_intercept + linear_slope * x for x in grid_xs]
    network_grid = [network_predict(x)[0] for x in grid_xs]

    ax.plot(grid_xs, linear_grid, linestyle="--", color="tab:red", label="linear baseline")
    ax.plot(grid_xs, network_grid, color="tab:green", label="single-hidden-layer network")
    ax.set_xlabel("feature_x [unitless]")
    ax.set_ylabel("target_y [unitless]")
    ax.set_title("Neural Network Basics: Nonlinear Regression Demo")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
try:
    import torch
except ModuleNotFoundError as exc:
    print(f"Optional PyTorch comparison skipped: {exc}")
else:
    torch.manual_seed(0)
    model = torch.nn.Sequential(
        torch.nn.Linear(1, hidden_size),
        torch.nn.Tanh(),
        torch.nn.Linear(hidden_size, 1),
    )
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = torch.nn.MSELoss()

    x_train_tensor = torch.tensor([[item[0]] for item in standardized_train], dtype=torch.float32)
    y_train_tensor = torch.tensor([[item[1]] for item in standardized_train], dtype=torch.float32)
    x_test_tensor = torch.tensor([[item[0]] for item in standardized_test], dtype=torch.float32)

    for _ in range(6000):
        optimizer.zero_grad()
        prediction_tensor = model(x_train_tensor)
        loss = loss_fn(prediction_tensor, y_train_tensor)
        loss.backward()
        optimizer.step()

    torch_predictions = model(x_test_tensor).detach().flatten().tolist()
    print("PyTorch test RMSE:", round(root_mean_squared_error(test_ys, torch_predictions), 4))
    print("PyTorch predictions:", [round(value, 3) for value in torch_predictions])
